# Composite elements — one user element, many atoms

A **composite element** presents as a single element but expands, at build time, into a small graph of atomic elements joined by internal edges.
The solver, Jacobian and perturbation layers never see a composite — at solve time the expanded graph is indistinguishable from a hand-built one (no new kernels).
The user's declared node/edge ids are preserved (internals append at the tail), so existing edge-indexed analysis keeps working unchanged.

Two families are shown here:

- **Class-1 macros** — a fixed recipe of heterogeneous atoms (`orifice`, `lossy_nozzle`, `helmholtz_resonator`, `sudden_contraction`).
- **Class-2 discretizations** — an `N`-segment chain that resolves a continuous element (`fanno_pipe`, `tapered_duct`), with `N` a convergence knob.

In [ ]:
import warnings
import numpy as np
import plotly.graph_objects as go

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.shell import build_problem
from nefes.elements.composite import grid_refine
from nefes.thermo.configure import perfect_gas
from nefes.perturbation import perturbation_response
from nefes.assembly.recover import ES_M, ES_P, ES_PT, ES_AREA
from nefes.plotting import use_nefes_theme

_ = use_nefes_theme()

GAMMA, R = 1.4, 287.0
GAS = perfect_gas(R, GAMMA)
CP = GAMMA * R / (GAMMA - 1.0)
P0, T0 = 101325.0, 300.0

## 1 — the orifice round-trips a hand-placed `iac + sac`

The De Domenico orifice is an isentropic contraction to the throat followed by a Borda re-expansion.
`cat.orifice(throat_area)` is exactly that assembly behind one element.
We solve it both ways and confirm the mean flow on the shared inlet/outlet edges and the compact scattering matrix are identical.

In [ ]:
A1, AT, A2 = 3.0e-3, 1.0e-3, 3.0e-3
PT_IN = 140000.0

#  (a) hand-built: inlet - iac - sac - outlet
hand = [cat.total_pressure_inlet(PT_IN, T0), cat.isentropic_area_change(), cat.sudden_area_change(), cat.pressure_outlet(P0, T0)]
ph = build_problem(GAS, hand, [(0, 1, A1), (1, 2, AT), (2, 3, A2)], 1.0, P0, CP * T0)
from nefes.solver import solve
sh = solve(ph)

#  (b) composite: inlet - orifice - outlet
net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
i = net.add(cat.total_pressure_inlet(PT_IN, T0))
orf = net.add(cat.orifice(AT, name="orifice"))
o = net.add(cat.pressure_outlet(P0, T0))
net.connect(i, orf, A1)
net.connect(orf, o, A2)
sc = net.solve()

from nefes.solver.report import states_table
eh, ec = states_table(ph, sh.x), sc.table()
S_hand = perturbation_response(ph, sh.x, np.array([120.0])).acoustic_scattering_matrix(0, 2)[0]
S_comp = perturbation_response(sc.problem, sc.x, np.array([120.0])).acoustic_scattering_matrix(0, 1)[0]
print(f"inlet Mach   hand {eh[ES_M, 0]:.5f}   composite {ec[ES_M, 0]:.5f}")
print(f"throat Mach  hand {eh[ES_M, 1]:.5f}   composite {ec[ES_M, sc.composite(orf).throat]:.5f}")
print(f"outlet Mach  hand {eh[ES_M, 2]:.5f}   composite {ec[ES_M, 1]:.5f}")
print(f"scattering matrix max |difference|: {np.max(np.abs(S_hand - S_comp)):.2e}")

**Result.** Identical to solver tolerance — the composite is a pure build-time convenience.
`solution.composite('orifice').throat` reads the throat state directly, without knowing the expanded edge layout.

## 2 — the Helmholtz resonator as one element

`cat.helmholtz_resonator(volume, neck_length, neck_area)` wraps the tee + neck duct + cavity side-branch behind one element (the TODO's first user of composites).
It resonates at the lumped Helmholtz frequency, shorting the main line into a transmission-loss peak.

In [ ]:
V, AN, LN, AM, LM = 1.0e-3, 5.0e-4, 0.02, 3.0e-3, 0.05
freqs = np.linspace(50.0, 1100.0, 1100)

net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
i = net.add(cat.total_pressure_inlet(P0, T0))
d1 = net.add(cat.duct(LM))
hr = net.add(cat.helmholtz_resonator(V, LN, AN, name="resonator"))
d2 = net.add(cat.duct(LM))
o = net.add(cat.pressure_outlet(P0, T0))
net.connect(i, d1, AM)
net.connect(d1, hr, AM)
net.connect(hr, d2, AM)
e_out = net.connect(d2, o, AM)
sol = net.solve()

resp = perturbation_response(sol.problem, sol.x, freqs)
with warnings.catch_warnings():  # inlet/outlet straddle the internal tee (expected)
    warnings.simplefilter("ignore")
    tau = resp.acoustic_scattering_matrix(0, e_out)[:, 1, 0]
tl = -20.0 * np.log10(np.abs(tau))
c = sol.field("c")[0]
f0 = c * np.sqrt(AN / (V * LN)) / (2.0 * np.pi)

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs, y=tl, name="transmission loss", line=dict(color="#1f77b4", width=2)))
fig.add_vline(x=f0, line=dict(color="#d62728", width=1, dash="dot"), annotation_text="f0", annotation_position="top")
fig.update_layout(title="helmholtz_resonator — one element, a Helmholtz peak",
                  xaxis_title="Frequency, Hz", yaxis_title="Transmission Loss, dB", width=820, height=440)
fig.show()
print(f"analytic f0 = {f0:.1f} Hz   Nefes peak = {freqs[int(np.argmax(tl))]:.1f} Hz")

**Result.** The composite lands the resonance on the analytic Helmholtz frequency, exactly as the hand-built tee + neck + cavity would.

## 3 — `fanno_pipe`: a discretized pipe resolves the Mach rise

A long pipe is Fanno flow — wall friction accelerates the subsonic flow toward the exit, so a single lumped friction coefficient misses the gradient.
`cat.fanno_pipe(length, diameter, friction, N)` chains `N` pipe atoms; as `N` grows the chain converges to the true Fanno solution.

In [ ]:
L, D, F, MDOT = 8.0, 0.05, 0.03, 0.55
area = np.pi * D**2 / 4.0


def fanno(N):
    els = [cat.mass_flow_inlet(MDOT, T0), cat.fanno_pipe(L, D, F, N), cat.pressure_outlet(P0, T0)]
    prob = build_problem(GAS, els, [(0, 1, area), (1, 2, area)], MDOT, P0, CP * T0)
    return prob, solve(prob)


fig = go.Figure()
for N, col in zip((2, 8, 32), ("#9ecae1", "#4292c6", "#08519c")):
    prob, s = fanno(N)
    est = states_table(prob, s.x)
    internal = sorted(prob.composite_map.internal_edges)
    edges = [0] + internal + [1]  # inlet, interior (flow order), outlet
    x = np.linspace(0.0, L, len(edges))
    fig.add_trace(go.Scatter(x=x, y=[est[ES_M, e] for e in edges], mode="lines+markers", name=f"N = {N}", line_color=col))
fig.update_layout(title="fanno_pipe — the Mach profile resolves as N grows",
                  xaxis_title="Axial position, m", yaxis_title="Mach number", width=820, height=440)
fig.show()

def fanno_table(N):
    prob, s = fanno(N)
    return states_table(prob, s.x)


#  probe the resolving quantities -- the inlet Mach and the integrated friction loss
gr = grid_refine(fanno_table, 16, lambda e: {"M_in": float(e[ES_M, 0]), "pt_drop": float(e[ES_PT, 0] - e[ES_PT, 1])})
print(f"grid refinement N=16 -> 32: quantities change at most {gr.worst:.2e} (converged: {gr.converged()})")

**Result.** Each segment carries a locally-uniform mean state, so the chain captures the continuous Mach rise — and a doubling of `N` barely moves the resolving quantities (the inlet Mach, the integrated loss), which is the built-in convergence check (`grid_refine`).

## 4 — `tapered_duct`: a con-di nozzle chokes at its true throat

`cat.tapered_duct(table)` discretizes a continuously area-varying passage into compact area-change + duct segments.
The standard input is a table of `(x, A)` pairs — the axial positions `x` [m] set the station spacing (which **may be non-uniform**, e.g. clustered near a throat) and the total length is inferred from them; a callable `A(x)` is also accepted.
Each segment carries a real `duct`, so the taper propagates waves through its interior.
A converging-diverging profile chokes at its narrowest station — the throat — which the composite's `throat` edge identifies.

In [ ]:
A_in, A_th, A_out, Ln = 3.0e-3, 1.5e-3, 3.0e-3, 0.4
N = 16
half = N // 2
areas = list(np.linspace(A_in, A_th, half + 1)) + list(np.linspace(A_th, A_out, N - half + 1))[1:]
table = list(zip(np.linspace(0.0, Ln, len(areas)), areas))  # (x, A) pairs; length inferred from x

net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
i = net.add(cat.total_pressure_inlet(108000.0, T0))
td = net.add(cat.tapered_duct(table, name="nozzle"))
o = net.add(cat.pressure_outlet(P0, T0))
net.connect(i, td, A_in)
net.connect(td, o, A_out)
sol = net.solve()
est = sol.table()
cv = sol.composite(td)

#  order the composite's edges along the axis by their station area pattern (contract then expand)
edges = [0] + sorted(cv.internal_edges) + [1]
xs = np.linspace(0.0, Ln, len(edges))
M = [est[ES_M, e] for e in edges]
A = [est[ES_AREA, e] for e in edges]

fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=M, mode="lines+markers", name="Mach", line=dict(color="#08519c", width=2), yaxis="y1"))
fig.add_trace(go.Scatter(x=xs, y=np.array(A) * 1e3, name="area [10⁻³ m²]", line=dict(color="#bdbdbd", width=2, dash="dash"), yaxis="y2"))
fig.update_layout(title="tapered_duct — the con-di throat is the fastest, narrowest edge",
                  xaxis_title="Axial position, m",
                  yaxis=dict(title="Mach number"),
                  yaxis2=dict(title="area, 10⁻³ m²", overlaying="y", side="right", showgrid=False),
                  width=820, height=440)
fig.show()
print(f"throat edge {cv.throat}:  area {est[ES_AREA, cv.throat]:.2e} (min), Mach {est[ES_M, cv.throat]:.4f} (max, subsonic)")

**Result.** The resolved Mach peaks at the geometric throat — the min-area station — with the isentropic-area-change complementarity engaging on exactly that segment.
Pushing the inlet pressure further drives the throat to `M = 1`, the v1 (subsonic) choke limit.

## 5 — `sudden_contraction`: the resolved vena contracta

A sudden contraction necks to a **vena contracta** of area `cc * downstream_area`, where the static pressure bottoms out, then re-expands with loss.
`cat.sudden_contraction(downstream_area, cc)` resolves that state (isentropic to the vena contracta + Borda re-expansion), so the minimum static pressure and the compressible loss are exact — unlike the lumped `O(M²)` cc-loss head.

In [ ]:
A_BIG, A_SMALL, CC = 4.0e-3, 1.0e-3, 0.62


def contraction(pt_in):
    net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
    i = net.add(cat.total_pressure_inlet(pt_in, T0))
    sc = net.add(cat.sudden_contraction(A_SMALL, cc=CC, name="contr"))
    o = net.add(cat.pressure_outlet(P0, T0))
    a = net.connect(i, sc, A_BIG)
    b = net.connect(sc, o, A_SMALL)
    return net.solve(), a, b


def sac_loss(pt_in):  # the incompressible O(M^2) reference
    net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
    i = net.add(cat.total_pressure_inlet(pt_in, T0))
    s = net.add(cat.sudden_area_change(cc=CC))
    o = net.add(cat.pressure_outlet(P0, T0))
    a = net.connect(i, s, A_BIG)
    b = net.connect(s, o, A_SMALL)
    e = net.solve().table()
    return (e[ES_PT, a] - e[ES_PT, b]) / e[ES_PT, a], float(e[ES_M, b])


#  the static-pressure dip at the vena contracta
sol, a, b = contraction(130000.0)
est = sol.table()
th = sol.composites[0].throat
fig = go.Figure()
fig.add_trace(go.Scatter(x=["inlet", "vena contracta", "outlet"], y=[est[ES_P, a] / 1e5, est[ES_P, th] / 1e5, est[ES_P, b] / 1e5],
                         mode="lines+markers", line=dict(color="#08519c", width=2)))
fig.update_layout(title="sudden_contraction — static pressure bottoms at the vena contracta",
                  yaxis_title="Static pressure, bar", width=820, height=400)
fig.show()

#  compressible accuracy: the resolved loss diverges from the O(M^2) model as Mach rises
pts = np.linspace(102000.0, 130000.0, 12)
res = [(sac_loss(p), contraction(p)) for p in pts]
m_out = [r[0][1] for r in res]
loss_sac = [r[0][0] for r in res]
loss_res = []
for _ls, (s, aa, bb) in res:
    e = s.table()
    loss_res.append((e[ES_PT, aa] - e[ES_PT, bb]) / e[ES_PT, aa])
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=m_out, y=loss_sac, name="sudden_area_change  (incompressible, O(M²))", line=dict(color="#bdbdbd", width=2, dash="dash")))
fig2.add_trace(go.Scatter(x=m_out, y=loss_res, name="sudden_contraction  (resolved vena contracta)", line=dict(color="#08519c", width=2)))
fig2.update_layout(title="Compressible loss correction grows with Mach",
                   xaxis_title="Downstream Mach", yaxis_title="Total-pressure loss fraction", width=820, height=440)
fig2.show()

**Result.** The vena contracta carries the lowest static pressure in the element — the cavitation-relevant minimum a lumped loss cannot report.
The two loss models agree at low Mach and separate as the flow compresses, the resolved composite being exact within the v1 subsonic range.

## Takeaway

| composite | class | expands to | reads back |
|-----------|-------|------------|------------|
| `orifice`, `lossy_nozzle` | 1 macro | `iac (+ iac) + sac` | throat state |
| `helmholtz_resonator` | 1 macro | tee + neck duct + cavity | resonance |
| `sudden_contraction` | 1 macro | isentropic + Borda | vena-contracta state |
| `fanno_pipe` | 2 discretization | N × `pipe` atom | Mach profile, converges in N |
| `tapered_duct` | 2 discretization | N × (`iac` + `duct`) from an (x, A) table | choke at the true throat |

Every recipe rides the same element-agnostic expander; the solved network is byte-identical to a hand build, and `solution.composite(name)` projects the result back to the user-facing element.

## Export for FNetLibUI

Composite elements serialize to the UI format as the **single node the user specified** -- an `orifice` saves as one `Orifice` node carrying its `throatArea`, never its expanded internals -- and the UI's *Composite elements* palette offers the same six elements, so a saved case round-trips both ways.
Save with `sol.to_yaml(path)` (mean-flow results embedded) or `network.to_yaml(path)` (topology only), then open the file in FNetLibUI; loading a UI-saved case back is `nefes.io.load_case`.
The cell below exports the composite orifice line from Section 1 directly.


In [ ]:
import os, tempfile

#  composites serialize as themselves: this case carries a single Orifice node
_out = os.path.join(tempfile.mkdtemp(), "composite_elements.yaml")
sc.to_yaml(_out)  # embeds the mean-flow results; use sc.network.to_yaml(_out) for topology only
print("saved case:", _out)